# IT5022 - Fundamentals of Machine Learning
## Notebook 02: Leakage-Safe Data Preprocessing & Feature Engineering
**Project:** Real-World Credit Risk Scoring & Customer Financial Segmentation  
**Authors:** MSc Candidate Ginuka (MS26922898) 

---

### Objectives of this Notebook:
1. **Enforce Leakage-Free Design:** Create an 80/20 Stratified Train-Test Split *before* any transformation.
2. **Apply Domain-Specific Transformations:**
   - Derive missing `Monthly_Income` from `Annual_Income / 12`.
   - Parse `Credit_History_Age` text into clean numerical years.
   - Extract loan counts and specific loan-type risk indicators (e.g., Payday loans) from `Type_of_Loan`.
   - Encode ordinal feature `Credit_Mix` and binary flag `Payment_of_Min_Amount`.
3. **Construct Modular ColumnTransformer:**
   - Numerical Imputation (Median) + StandardScaler.
   - Categorical Imputation (Most Frequent) + OneHotEncoder.
4. **Export Clean Processed Datasets** for subsequent modeling.


In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Import local pipeline modules
sys.path.append(os.path.abspath(".."))
from src.data_pipeline import CreditDomainFeatureEngineer, build_preprocessor, get_feature_lists

print("Pipeline modules loaded successfully.")


Pipeline modules loaded successfully.


### 1. Ingest Raw Dataset & Stratified Split
We partition the dataset into 80% Training ($n=6,953$) and 20% Test ($n=1,739$) stratified on `Credit_Score`.


In [2]:
data_path = os.path.join("..", "dataset", "dataset.csv")
df = pd.read_csv(data_path)

# Partition predictors and binary target (1 = Poor, 0 = Good)
X = df.drop(columns=['Credit_Score'])
y = (df['Credit_Score'] == 'Poor').astype(int)  # 1 = Poor (Risk), 0 = Good

# Stratified 80/20 train/test partition preserving class ratio to prevent data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Instances: {X_train_raw.shape[0]} (Poor prevalence: {y_train.mean():.2%})")
print(f"Testing Instances:  {X_test_raw.shape[0]} (Poor prevalence: {y_test.mean():.2%})")


Training Instances: 6953 (Poor prevalence: 59.66%)
Testing Instances:  1739 (Poor prevalence: 59.63%)


### 2. Domain Feature Engineering
Using `CreditDomainFeatureEngineer` to apply domain logic consistently.


In [3]:
engineer = CreditDomainFeatureEngineer(drop_identifiers=True)

X_train_eng = engineer.transform(X_train_raw)
X_test_eng = engineer.transform(X_test_raw)

print(f"Features after domain engineering: {X_train_eng.shape[1]}")
X_train_eng.head(3)


Features after domain engineering: 15


,Age,Annual_Income,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Payment_of_Min_Amount,Amount_invested_monthly,Payment_Behaviour,Credit_History_Years,Has_Payday_Loan,Has_Mortgage,Credit_Mix_Ordinal
6334,40.0,17594.015,0.0,3.0,11.0,0.0,2,13.0,0.0,89.990404,Low_spent_Small_value_payments,28.0,0,0,2.0
3799,45.0,54346.350,NaN,7.0,19.0,8.0,51,25.0,1.0,33.510037,NaN,12.0,1,1,NaN
50,28.0,32595.220,3.0,3.0,7.0,2.0,10,7.0,0.0,154.258853,High_spent_Small_value_payments,16.0,0,0,NaN


### 3. Build & Fit ColumnTransformer (Zero Data Leakage)
We construct the numerical and categorical transformers, fitting **only** on `X_train_eng`.


In [4]:
numeric_cols, categorical_cols = get_feature_lists(X_train_eng)
print("Numeric Features:", numeric_cols)
print("Categorical Features:", categorical_cols)

preprocessor = build_preprocessor(numeric_cols, categorical_cols)

# Fit transformer strictly on training split to guarantee zero test leakage
X_train_proc = preprocessor.fit_transform(X_train_eng)
X_test_proc = preprocessor.transform(X_test_eng)

# Extract one-hot encoded column names
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols).tolist()
all_feature_names = numeric_cols + cat_feature_names

print(f"Total features after One-Hot Encoding: {X_train_proc.shape[1]}")
print(f"X_train_proc shape: {X_train_proc.shape}")
print(f"X_test_proc shape:  {X_test_proc.shape}")


Numeric Features: ['Age', 'Annual_Income', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Payment_of_Min_Amount', 'Amount_invested_monthly', 'Credit_History_Years', 'Has_Payday_Loan', 'Has_Mortgage', 'Credit_Mix_Ordinal']
Categorical Features: ['Payment_Behaviour']
Total features after One-Hot Encoding: 20
X_train_proc shape: (6953, 20)
X_test_proc shape:  (1739, 20)


### 4. Verify Feature Processing Integrity
We ensure zero NaNs exist and all matrices are properly scaled.


In [5]:
assert not np.isnan(X_train_proc).any(), "NaN values found in training set!"
assert not np.isnan(X_test_proc).any(), "NaN values found in test set!"

train_means = np.mean(X_train_proc[:, :len(numeric_cols)], axis=0)
train_stds = np.std(X_train_proc[:, :len(numeric_cols)], axis=0)

print("Sample Numeric Feature Means (should be ~0.0):", np.round(train_means[:4], 3))
print("Sample Numeric Feature Stds  (should be ~1.0):", np.round(train_stds[:4], 3))
print("Data preprocessing and feature engineering successfully verified!")


Sample Numeric Feature Means (should be ~0.0): [ 0. -0. -0.  0.]
Sample Numeric Feature Stds  (should be ~1.0): [1. 1. 1. 1.]
Data preprocessing and feature engineering successfully verified!
